## 1. Repository audit (observed, not assumed)

| Requirement | Uploaded repository status | This notebook |
|---|---|---|
| Profile schema | Backend `nutrition_profiles` package is empty; no final API DTO | Proposed validated contract |
| Profile to reasoning context | `RuleContext` exists with strict digest/version rules; mapping from UI profile absent | Boundary + explicit adapter obligation |
| Recipe filtering | `filter_candidates` implemented | Typed adapter |
| Combination evaluation | `evaluate_combination` implemented | Typed adapter |
| Daily CSP | `solve_daily_plan` implemented (3 slots, bounded search) | Typed adapter |
| Weekly 21-meal CSP | No weekly solver found in uploaded snapshot | No fake solver; demonstration only |
| Accepted KB, source identity | Knowledge snapshots, manifests, digest checking implemented in AI layer | Trusted release registry boundary |
| Backend endpoints | FastAPI skeleton, health only; no meal-plan endpoints | Proposed request/response contracts |
| Storage, swaps, regeneration | No repository service/DB implementation found | Local SQLite reference implementation |

Existing AI code: `apps/ai_system/src/savora_ai/core/reasoning/` and `apps/ai_system/src/savora_ai/schemas/`. Backend skeleton: `apps/backend/src/savora_backend/`. Proposed backend locations: `core/nutrition_profiles/`, `core/meal_plans/`, `contexts/` and router modules to be created. These are **proposals**, not existing routes.

## 2. Validated profile and planning inputs

In [45]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from uuid import uuid4
import json, sqlite3, hashlib, importlib.util

MEAL_SLOTS = ("breakfast", "lunch", "dinner")
VALID_DIETS = {"omnivore", "vegetarian", "vegan", "pescatarian"}
VALID_ALLERGENS = {"milk", "eggs", "fish", "shellfish", "tree_nuts", "peanuts", "wheat", "soy", "sesame"}

def require_string_list(value: Any, label: str) -> list[str]:
    if not isinstance(value, list) or any(not isinstance(x, str) or not x.strip() for x in value):
        raise ValueError(f"{label} must be a list of nonempty strings")
    return list(dict.fromkeys(x.strip().lower() for x in value))

@dataclass
class UserProfile:
    user_id: str
    dietary_preference: str = "omnivore"
    restrictions: list[str] = field(default_factory=list)
    allergies: list[str] = field(default_factory=list)
    disliked_ingredients: list[str] = field(default_factory=list)
    preferred_cuisines: list[str] = field(default_factory=list)
    calorie_target_kcal: int | None = None
    protein_target_g: float | None = None
    carb_target_g: float | None = None
    fat_target_g: float | None = None
    max_prep_minutes: int | None = None
    budget_per_day: float | None = None
    household_size: int = 1
    timezone: str = "UTC"

    def __post_init__(self):
        if not isinstance(self.user_id, str) or not self.user_id.strip():
            raise ValueError("user_id is required")
        if self.dietary_preference not in VALID_DIETS:
            raise ValueError(f"Unsupported dietary_preference: {self.dietary_preference}")
        for name in ("restrictions", "allergies", "disliked_ingredients", "preferred_cuisines"):
            setattr(self, name, require_string_list(getattr(self, name), name))
        if not set(self.allergies).issubset(VALID_ALLERGENS):
            raise ValueError("Unknown allergen; use canonical allergen names")
        for name in ("calorie_target_kcal", "protein_target_g", "carb_target_g", "fat_target_g", "max_prep_minutes", "budget_per_day"):
            v = getattr(self, name)
            if v is not None and (isinstance(v, bool) or not isinstance(v, (int, float)) or not 0 < v < float("inf")):
                raise ValueError(f"{name} must be a finite positive number")
        if not isinstance(self.household_size, int) or isinstance(self.household_size, bool) or self.household_size < 1:
            raise ValueError("household_size must be a positive integer")
        if not self.timezone:
            raise ValueError("timezone is required")

profile = UserProfile(user_id="demo-user", dietary_preference="vegetarian", allergies=["peanuts"],
                      calorie_target_kcal=2100, protein_target_g=90, max_prep_minutes=45)
print(json.dumps(asdict(profile), indent=2))

{
  "user_id": "demo-user",
  "dietary_preference": "vegetarian",
  "restrictions": [],
  "allergies": [
    "peanuts"
  ],
  "disliked_ingredients": [],
  "preferred_cuisines": [],
  "calorie_target_kcal": 2100,
  "protein_target_g": 90,
  "carb_target_g": null,
  "fat_target_g": null,
  "max_prep_minutes": 45,
  "budget_per_day": null,
  "household_size": 1,
  "timezone": "UTC"
}


## 3. Accepted knowledge releases and trusted source registry

The real AI layer uses cryptographically bound snapshot models (`KnowledgeBaseSnapshot`, `RuleContext`, candidate and daily-source snapshots). **Never create a fake digest to satisfy their validators.** Production loaders must select an approved KB by ID/version/digest from a trusted database or release registry, then deserialize and validate it using the real Pydantic DTOs. Do not trust a client-supplied KB body or digest.

In [46]:
@dataclass(frozen=True)
class ReleaseIdentity:
    kb_id: str
    kb_version: str
    kb_digest: str
    source_id: str
    source_version: str

    def __post_init__(self):
        for key, val in asdict(self).items():
            if not isinstance(val, str) or not val.strip():
                raise ValueError(f"{key} must be nonempty")

class ApprovedReleaseRegistry:
    def __init__(self):
        self._approved: dict[tuple[str, str], ReleaseIdentity] = {}
    def approve(self, identity: ReleaseIdentity) -> None:
        self._approved[(identity.kb_id, identity.kb_version)] = identity
    def resolve(self, kb_id: str, version: str, expected_digest: str) -> ReleaseIdentity:
        record = self._approved.get((kb_id, version))
        if record is None or record.kb_digest != expected_digest:
            raise ValueError("Unapproved knowledge release or digest mismatch")
        return record

registry = ApprovedReleaseRegistry()
demo_release = ReleaseIdentity("demo-kb", "demo-v1", "demo-digest-not-a-real-kb-hash", "demo-source", "v1")
registry.approve(demo_release)
assert registry.resolve(demo_release.kb_id, demo_release.kb_version, demo_release.kb_digest) == demo_release
print("Local release registry validation passed (demo only; not an accepted production KB).")

Local release registry validation passed (demo only; not an accepted production KB).


## 4. API request/response contracts (proposed)

Suggested routes: `PUT /v1/profiles/{user_id}`, `POST /v1/meal-plans`, `GET /v1/meal-plans/{plan_id}`, `POST /v1/meal-plans/{plan_id}/swaps`, `POST /v1/meal-plans/{plan_id}/regenerate`. They are **not currently implemented** in the uploaded FastAPI skeleton. Backend must derive authenticated user identity server-side and enforce plan ownership.

In [47]:
@dataclass
class PlanRequest:
    user_id: str
    start_date: str
    days: int
    profile: UserProfile
    release: ReleaseIdentity
    mode: str = "weekly"
    exclude_recipe_ids: list[str] = field(default_factory=list)

    def __post_init__(self):
        from datetime import date
        date.fromisoformat(self.start_date)
        if self.days != 7 or self.mode != "weekly":
            raise ValueError("21-meal contract requires exactly seven days")
        if self.user_id != self.profile.user_id:
            raise ValueError("Profile owner and plan owner mismatch")
        self.exclude_recipe_ids = require_string_list(self.exclude_recipe_ids, "exclude_recipe_ids")

@dataclass
class MealAssignment:
    day: int
    slot: str
    recipe_id: str
    recipe_name: str
    status: str
    explanation: str

    def __post_init__(self):
        if self.day not in range(1, 8) or self.slot not in MEAL_SLOTS:
            raise ValueError("Invalid day or slot")
        if not self.recipe_id or not self.recipe_name or not self.explanation:
            raise ValueError("Missing meal assignment fields")
        if self.status not in {"pass", "fail", "unknown", "not_evaluated"}:
            raise ValueError("Invalid constraint status")

@dataclass
class PlanResponse:
    plan_id: str
    user_id: str
    revision: int
    status: str
    release: ReleaseIdentity
    meals: list[MealAssignment]
    constraint_results: list[dict[str, Any]]
    explanations: list[str]
    generated_at: str
    validated_by_csp: bool = False

    def __post_init__(self):
        if len(self.meals) != 21 or len({(m.day, m.slot) for m in self.meals}) != 21:
            raise ValueError("Weekly plan must contain 21 unique day/slot assignments")
        if self.validated_by_csp and any(m.status != "pass" for m in self.meals):
            raise ValueError("CSP-validated plan cannot contain non-passing meals")

request = PlanRequest(user_id=profile.user_id, start_date="2026-10-19", days=7, profile=profile, release=demo_release)
print(json.dumps({"endpoint": "POST /v1/meal-plans", "request_example": asdict(request)}, indent=2))

{
  "endpoint": "POST /v1/meal-plans",
  "request_example": {
    "user_id": "demo-user",
    "start_date": "2026-10-19",
    "days": 7,
    "profile": {
      "user_id": "demo-user",
      "dietary_preference": "vegetarian",
      "restrictions": [],
      "allergies": [
        "peanuts"
      ],
      "disliked_ingredients": [],
      "preferred_cuisines": [],
      "calorie_target_kcal": 2100,
      "protein_target_g": 90,
      "carb_target_g": null,
      "fat_target_g": null,
      "max_prep_minutes": 45,
      "budget_per_day": null,
      "household_size": 1,
      "timezone": "UTC"
    },
    "release": {
      "kb_id": "demo-kb",
      "kb_version": "demo-v1",
      "kb_digest": "demo-digest-not-a-real-kb-hash",
      "source_id": "demo-source",
      "source_version": "v1"
    },
    "mode": "weekly",
    "exclude_recipe_ids": []
  }
}


## 5. AI engine adapters — actual repository signatures

These adapters import the repository's real Pydantic contracts and functions. They **require genuine validated** knowledge/context/candidate/source objects, and intentionally do not synthesize them from incomplete profile information. Profile-to-`RuleContext` mapping must be implemented by the knowledge/context owner after canonical facts and accepted release identities are agreed.

In [48]:
def find_repo_root(start: Path | None = None) -> Path | None:
    for candidate in [start or Path.cwd(), *((start or Path.cwd()).parents)]:
        if (candidate / "apps" / "ai_system" / "src" / "savora_ai").is_dir():
            return candidate
    return None

def load_real_ai_interfaces(repo_root: Path | None = None) -> dict[str, Any]:
    import sys
    root = repo_root or find_repo_root()
    if root is None:
        raise FileNotFoundError("Open this notebook from inside the cloned Savora repository")
    ai_src = root / "apps" / "ai_system" / "src"
    if str(ai_src) not in sys.path:
        sys.path.insert(0, str(ai_src))
    from savora_ai.core.reasoning.filtering import filter_candidates
    from savora_ai.core.reasoning.combinations import evaluate_combination
    from savora_ai.core.reasoning.daily import solve_daily_plan, build_day_check_manifest
    from savora_ai.core.reasoning.kb import build_knowledge_snapshot
    from savora_ai.schemas.candidates import RuleContext, CandidateRecipeFacts
    from savora_ai.schemas.combinations import CombinationFacts
    from savora_ai.schemas.daily import DailyPlanningRequest, DailySourceSnapshot
    from savora_ai.schemas.daily_solver import DailySearchLimits
    from savora_ai.schemas.knowledge_base import KnowledgeBaseSnapshot
    return dict(filter_candidates=filter_candidates, evaluate_combination=evaluate_combination,
                solve_daily_plan=solve_daily_plan, build_day_check_manifest=build_day_check_manifest,
                build_knowledge_snapshot=build_knowledge_snapshot, RuleContext=RuleContext,
                CandidateRecipeFacts=CandidateRecipeFacts, CombinationFacts=CombinationFacts,
                DailyPlanningRequest=DailyPlanningRequest, DailySourceSnapshot=DailySourceSnapshot,
                DailySearchLimits=DailySearchLimits, KnowledgeBaseSnapshot=KnowledgeBaseSnapshot)

def run_recipe_filter(candidates, knowledge, context, interfaces):
    return interfaces["filter_candidates"](tuple(candidates), knowledge=knowledge, context=context)

def run_combination_evaluation(combination, candidates, knowledge, context, interfaces):
    return interfaces["evaluate_combination"](combination, candidates=tuple(candidates), knowledge=knowledge, context=context)

def run_daily_csp(request, candidates, knowledge, context, sources, interfaces, limits=None):
    kwargs = dict(candidates=tuple(candidates), knowledge=knowledge, context=context, sources=sources)
    if limits is not None:
        kwargs["limits"] = limits
    return interfaces["solve_daily_plan"](request, **kwargs)

root = find_repo_root()
if root is None:
    print("Standalone contract demo mode. Open notebook inside repository to enable real AI imports.")
else:
    try:
        ai_interfaces = load_real_ai_interfaces(root)
        print("Real Savora AI interfaces imported:", sorted(ai_interfaces))
    except (ImportError, ModuleNotFoundError) as exc:
        print("Real AI import unavailable; install project dependencies:", exc)

Real Savora AI interfaces imported: ['CandidateRecipeFacts', 'CombinationFacts', 'DailyPlanningRequest', 'DailySearchLimits', 'DailySourceSnapshot', 'KnowledgeBaseSnapshot', 'RuleContext', 'build_day_check_manifest', 'build_knowledge_snapshot', 'evaluate_combination', 'filter_candidates', 'solve_daily_plan']


## 6. How profile data enters AI reasoning (required integration boundary)

**Flow:** authenticated user → validate `UserProfile` → fetch approved KB release and source snapshots → map dietary/allergy/targets to canonical, evidence-backed facts → construct validated `RuleContext` with KB digest and context version → recipe filtering → combination evaluation → daily CSP → **weekly coordinator (not yet implemented)** → explanations/results → persistent storage.

Safety rule: unknown allergens, absent nutrition facts, or unresolved hard constraints must **not** be treated as passing. A weekly result is publishable only after all 21 assignments have been evaluated and required day/week constraints have passed. The repository currently has day-level reasoning, not an authoritative seven-day optimizer.

In [49]:
def require_authoritative_weekly_solver(solver=None):
    if solver is None:
        raise NotImplementedError(
            "No authoritative weekly 21-meal CSP solver was found in the uploaded repository. "
            "Do not publish the demonstration plan as constraint-validated."
        )
    return solver

print("Weekly CSP status: NOT IMPLEMENTED in supplied repository snapshot")

Weekly CSP status: NOT IMPLEMENTED in supplied repository snapshot


## 7. Contract-only 21-meal demonstration

The following uses placeholder recipe IDs and **not_evaluated** statuses solely to test serialization, uniqueness, persistence and swaps. It is not a recommendation or safe allergy/nutrition plan.

In [50]:
demo_meals = [MealAssignment(day=d, slot=s, recipe_id=f"placeholder-{d}-{s}",
                             recipe_name=f"Example {s.title()} {d}", status="not_evaluated",
                             explanation="Contract placeholder: no recipe facts or constraints evaluated")
              for d in range(1, 8) for s in MEAL_SLOTS]
demo_plan = PlanResponse(plan_id=str(uuid4()), user_id=profile.user_id, revision=1,
                         status="draft_unvalidated", release=demo_release, meals=demo_meals,
                         constraint_results=[{"status": "not_evaluated", "scope": "weekly"}],
                         explanations=["No weekly CSP run; no allergy or nutrition validation performed"],
                         generated_at=datetime.now(timezone.utc).isoformat(), validated_by_csp=False)
assert len(demo_plan.meals) == 21
print("Generated 21 contract placeholders; NOT a validated meal plan.")

Generated 21 contract placeholders; NOT a validated meal plan.


## 8. Storage, swap and regeneration (local reference only)

SQLite below is an isolated demo persistence layer. Production should use backend DB migrations, transactions, authenticated ownership checks, concurrency control, and full audit trails. Swap proposals must be re-evaluated by the actual reasoning engine before being accepted; the demo keeps them unvalidated.

In [51]:
class LocalPlanStore:
    def __init__(self, database_path=":memory:"):
        self.connection = sqlite3.connect(database_path)
        self.connection.execute("CREATE TABLE IF NOT EXISTS plans (plan_id TEXT PRIMARY KEY, user_id TEXT NOT NULL, revision INTEGER NOT NULL, payload TEXT NOT NULL)")
        self.connection.commit()
    def save(self, plan: PlanResponse):
        payload = json.dumps(asdict(plan), allow_nan=False)
        with self.connection:
            existing = self.connection.execute("SELECT revision FROM plans WHERE plan_id=?", (plan.plan_id,)).fetchone()
            if existing is not None and plan.revision <= existing[0]:
                raise ValueError("Revision must increase to prevent stale writes")
            self.connection.execute("INSERT INTO plans(plan_id,user_id,revision,payload) VALUES(?,?,?,?) ON CONFLICT(plan_id) DO UPDATE SET revision=excluded.revision,payload=excluded.payload", (plan.plan_id, plan.user_id, plan.revision, payload))
    def get(self, plan_id: str, user_id: str) -> PlanResponse:
        row = self.connection.execute("SELECT payload FROM plans WHERE plan_id=? AND user_id=?", (plan_id,user_id)).fetchone()
        if row is None: raise KeyError("Plan not found for this user")
        data = json.loads(row[0]); data["release"] = ReleaseIdentity(**data["release"])
        data["meals"] = [MealAssignment(**m) for m in data["meals"]]
        return PlanResponse(**data)

def propose_swap(plan: PlanResponse, day: int, slot: str, replacement_id: str, replacement_name: str) -> PlanResponse:
    if not replacement_id.strip() or not replacement_name.strip(): raise ValueError("Replacement ID/name required")
    data = asdict(plan)
    matches = [m for m in data["meals"] if m["day"] == day and m["slot"] == slot]
    if len(matches) != 1: raise ValueError("Unknown day/slot")
    target = matches[0]
    target.update(recipe_id=replacement_id, recipe_name=replacement_name, status="not_evaluated",
                  explanation="Swap proposed; requires full recipe/day/week constraint re-evaluation")
    data["revision"] += 1
    data["status"] = "draft_unvalidated"
    data["validated_by_csp"] = False
    data["constraint_results"] = [{"scope":"weekly", "status":"not_evaluated"}]
    data["explanations"] = ["Swap invalidates previous validation; rerun reasoning before publishing"]
    data["generated_at"] = datetime.now(timezone.utc).isoformat()
    data["release"] = ReleaseIdentity(**data["release"])
    data["meals"] = [MealAssignment(**m) for m in data["meals"]]
    return PlanResponse(**data)

def request_regeneration(old_plan: PlanResponse, exclude_recipe_ids: list[str]) -> dict:
    return {"user_id":old_plan.user_id, "previous_plan_id":old_plan.plan_id,
            "expected_revision":old_plan.revision, "exclude_recipe_ids":require_string_list(exclude_recipe_ids, "exclude_recipe_ids"),
            "status":"pending_solver", "message":"Invoke authoritative planner; do not reuse stale validations"}

store = LocalPlanStore()
store.save(demo_plan)
assert store.get(demo_plan.plan_id, profile.user_id).revision == 1
swapped = propose_swap(demo_plan, 1, "lunch", "new-placeholder", "Example replacement")
store.save(swapped)
assert store.get(demo_plan.plan_id, profile.user_id).revision == 2
print("Stored and retrieved draft; swap increments revision and invalidates constraints")
print(json.dumps(request_regeneration(swapped, ["placeholder-1-lunch"]), indent=2))

Stored and retrieved draft; swap increments revision and invalidates constraints
{
  "user_id": "demo-user",
  "previous_plan_id": "ab3cfb4b-2b64-4d7d-b833-bd240b773db2",
  "expected_revision": 2,
  "exclude_recipe_ids": [
    "placeholder-1-lunch"
  ],
  "status": "pending_solver",
  "message": "Invoke authoritative planner; do not reuse stale validations"
}


## 9. Example responses and proposed backend wiring

`POST /v1/meal-plans` should return `202 Accepted` with a job ID if asynchronous, or `201 Created` only after an authoritative plan exists. `GET` returns plan status and the 21 assignments only for complete plans; incomplete solver runs should return diagnostics, not invented meals. `POST /swaps` takes `{day, slot, replacement_recipe_id, expected_revision}` and rechecks all affected hard constraints. `POST /regenerate` takes `{expected_revision, excluded_recipe_ids, changed_profile?}` and creates a new revision/job. Return `409` on revision conflicts, `422` on invalid inputs or infeasible constraints, `403` on ownership violations. **These are proposed contracts, not live endpoints.**

In [52]:
proposed_contracts = {
 "PUT /v1/profiles/{user_id}": {"request": "UserProfile", "response": "UserProfile + version"},
 "POST /v1/meal-plans": {"request": "PlanRequest", "response": "202 {job_id, status} or 201 PlanResponse after verified CSP"},
 "GET /v1/meal-plans/{plan_id}": {"response": "PlanResponse (if complete) or job status/diagnostics"},
 "POST /v1/meal-plans/{plan_id}/swaps": {"request": {"day": 1, "slot":"lunch", "replacement_recipe_id":"recipe-id", "expected_revision":2}, "response":"revalidated PlanResponse or 422/409"},
 "POST /v1/meal-plans/{plan_id}/regenerate": {"request":{"expected_revision":2,"excluded_recipe_ids":[]}, "response":"202 {job_id, status}"}
}
print(json.dumps(proposed_contracts, indent=2))

{
  "PUT /v1/profiles/{user_id}": {
    "request": "UserProfile",
    "response": "UserProfile + version"
  },
  "POST /v1/meal-plans": {
    "request": "PlanRequest",
    "response": "202 {job_id, status} or 201 PlanResponse after verified CSP"
  },
  "GET /v1/meal-plans/{plan_id}": {
    "response": "PlanResponse (if complete) or job status/diagnostics"
  },
  "POST /v1/meal-plans/{plan_id}/swaps": {
    "request": {
      "day": 1,
      "slot": "lunch",
      "replacement_recipe_id": "recipe-id",
      "expected_revision": 2
    },
    "response": "revalidated PlanResponse or 422/409"
  },
  "POST /v1/meal-plans/{plan_id}/regenerate": {
    "request": {
      "expected_revision": 2,
      "excluded_recipe_ids": []
    },
    "response": "202 {job_id, status}"
  }
}


## 10. Automated contract smoke tests

In [53]:
def expect_error(fn, error_type=ValueError):
    try: fn()
    except error_type: return
    raise AssertionError(f"Expected {error_type.__name__}")

expect_error(lambda: UserProfile(user_id="x", allergies=["unknown_allergen"]))
expect_error(lambda: UserProfile(user_id="x", dietary_preference="invalid"))
expect_error(lambda: UserProfile(user_id="x", calorie_target_kcal=-1))
expect_error(lambda: PlanRequest(user_id="wrong", start_date="2026-10-19", days=7, profile=profile, release=demo_release))
expect_error(lambda: registry.resolve("demo-kb", "demo-v1", "wrong"))
expect_error(lambda: PlanResponse(plan_id="x", user_id="x", revision=1, status="complete", release=demo_release,
                                  meals=demo_meals[:-1], constraint_results=[], explanations=[], generated_at="now"))
expect_error(lambda: require_authoritative_weekly_solver(), NotImplementedError)
expect_error(lambda: store.save(swapped))
try:
    store.get(demo_plan.plan_id, "different-user")
except KeyError: pass
else: raise AssertionError("Owner isolation failed")
assert len({(m.day, m.slot) for m in swapped.meals}) == 21
assert swapped.validated_by_csp is False
print("PASS: schema, ownership, release validation, 21-slot uniqueness, revisions, swaps and solver guard")

PASS: schema, ownership, release validation, 21-slot uniqueness, revisions, swaps and solver guard


## 11. Remaining work before production integration

1. Agree final profile fields and canonical mappings with your teammate; align these proposals with actual UI/DB fields.
2. Implement authenticated backend routes and durable profile/plan repositories; the uploaded backend currently only provides a skeleton.
3. Implement trusted KB and source snapshot selection with exact version/digest validation; do not substitute the demo registry.
4. Build a **real** profile-to-`RuleContext` adapter using canonical, evidence-backed facts and release bindings.
5. Connect the real candidate/combination/daily reasoning DTOs to your data pipeline, with valid approved KB snapshots.
6. Implement and test the missing weekly CSP coordinator and week-level constraints before claiming a validated 21-meal plan.
7. Re-run affected recipe, day and week constraints for swaps/regeneration; preserve provenance and explanation traces.
8. Add API integration tests, real fixture data, DB migrations, authentication and CI validation.

**Notebook execution guarantee:** Local demonstration and contract smoke tests run without the repository or datasets. Real AI execution is conditional on installed dependencies, approved snapshots, actual candidate facts and the missing weekly coordinator; no guarantee of error-free production integration is possible from the supplied files alone.

# Part II — Qwen, NLU, entity normalization, dialogue and conversational UI

This section preserves the user's original conversational assistant, with portable dataset discovery and optional Qwen/interactive UI. It is a **separate conversational prototype**, not a wired FastAPI endpoint. Install `pandas` for recipe data. Qwen requires `transformers`, `torch`, and `accelerate`; Gradio/ipywidgets are optional. A dataset-free NLU smoke test is included, so the notebook can still execute without `RAW_recipes.csv`. Hard allergy enforcement and validated 21-meal planning must remain delegated to authoritative reasoning services.

In [54]:
# Optional dependency setup: uncomment in VS Code/Jupyter when needed.
# %pip install pandas numpy rapidfuzz
# %pip install transformers torch accelerate sentencepiece
# %pip install gradio ipywidgets


In [55]:
"""Savora: notebook-friendly Food.com conversational recipe assistant.
Qwen is optional; rule-based fallback works without a GPU or internet.
"""
import ast
import copy
import json
import re
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

ALIASES = {'potatoes':'potato','tomatoes':'tomato','onions':'onion','chillies':'chili',
           'chilies':'chili','capsicum':'bell pepper','bell peppers':'bell pepper',
           'brinjal':'eggplant','aubergine':'eggplant','garbanzo beans':'chickpeas',
           'chickpea':'chickpeas','coriander leaves':'cilantro','curd':'yogurt'}
DIET_ALIASES = {'gluten free':'gluten-free','dairy free':'dairy-free',
                'low carb':'low-carb','low fat':'low-fat','low sodium':'low-sodium',
                'plant based':'vegan'}
CUISINES = {'indian','italian','mexican','chinese','thai','japanese','french',
            'greek','korean','mediterranean','american','vietnamese'}
DIETS = {'vegetarian','vegan','gluten-free','dairy-free','low-carb','low-fat','low-sodium'}


def normalize(text):
    text = re.sub(r'\s+', ' ', str(text).lower().replace('_', ' ').strip())
    text = re.sub(r'[^\w\s-]', '', text)
    return ALIASES.get(text, DIET_ALIASES.get(text, text))


def parse_list(value):
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return []
    try:
        out = ast.literal_eval(value)
        return out if isinstance(out, list) else []
    except (SyntaxError, ValueError, TypeError, MemoryError):
        return []


@dataclass
class Slots:
    ingredients: list = field(default_factory=list)
    excluded: list = field(default_factory=list)
    cuisine: str | None = None
    diet: str | None = None
    max_minutes: int | None = None


class RecipeStore:
    def __init__(self, path, max_rows=None):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f'{path} not found. Set DATA_DIR to your CSV folder.')
        cols = ['id','name','minutes','ingredients','tags','steps','description']
        self.df = pd.read_csv(path, usecols=cols, nrows=max_rows, low_memory=False)
        self.df['minutes'] = pd.to_numeric(self.df['minutes'], errors='coerce')
        self.df['name'] = self.df['name'].fillna('Untitled recipe')
        for col in ('ingredients','tags','steps'):
            self.df[col] = self.df[col].map(parse_list)
        self.df['ingredient_set'] = self.df['ingredients'].map(
            lambda values: {normalize(v) for v in values})
        self.df['tag_set'] = self.df['tags'].map(
            lambda values: {normalize(v) for v in values})
        self.vocabulary = set().union(*self.df['ingredient_set'].tolist()) if len(self.df) else set()
        self.vocabulary.update(ALIASES)

    def search(self, slots, limit=5):
        df = self.df
        if slots.max_minutes is not None:
            df = df[df['minutes'].le(slots.max_minutes)]
        if slots.cuisine:
            df = df[df['tag_set'].map(lambda x: slots.cuisine in x)]
        if slots.diet:
            df = df[df['tag_set'].map(lambda x: slots.diet in x)]
        if slots.ingredients:
            df = df[df['ingredient_set'].map(lambda x: all(v in x for v in slots.ingredients))]
        if slots.excluded:
            df = df[df['ingredient_set'].map(lambda x: not any(v in x for v in slots.excluded))]
        return df.sort_values(['minutes','id']).head(limit).to_dict('records')


class NLU:
    def __init__(self, vocabulary, use_qwen=False, model_id='Qwen/Qwen2.5-1.5B-Instruct'):
        self.vocabulary = vocabulary
        self.pipe = None
        if use_qwen:
            try:
                import torch
                from transformers import pipeline
                kwargs = {'model': model_id, 'device_map': 'auto', 'torch_dtype': 'auto'}
                self.pipe = pipeline('text-generation', **kwargs)
            except Exception as exc:
                raise RuntimeError('Could not load Qwen. Install torch, transformers and accelerate; '
                                   'or use use_qwen=False. Original error: ' + str(exc)) from exc

    def _rules(self, utterance):
        text = normalize(utterance)
        if re.search(r'\b(reset|start over|clear preferences)\b', text):
            return {'intent':'reset'}
        if re.search(r'\b(help|what can you do)\b', text):
            return {'intent':'help'}
        cuisine = next((x for x in sorted(CUISINES, key=len, reverse=True)
                        if re.search(r'(?<!\w)'+re.escape(x)+r'(?!\w)', text)), None)
        diet = next((x for x in sorted(DIETS, key=len, reverse=True)
                     if re.search(r'(?<!\w)'+re.escape(x)+r'(?!\w)', text)), None)
        for alias, value in DIET_ALIASES.items():
            if re.search(r'(?<!\w)'+re.escape(alias)+r'(?!\w)', text):
                diet = value
        m = re.search(r'\b(?:under|less than|within|max(?:imum)?|up to|in)\s*(\d+)\s*(?:minutes?|mins?)\b', text)
        minutes = int(m.group(1)) if m else None
        positive, excluded = [], []
        for term in sorted(self.vocabulary, key=len, reverse=True):
            if len(term) < 3 or not re.search(r'(?<!\w)'+re.escape(term)+r'(?!\w)', text):
                continue
            canonical = normalize(term)
            if re.search(r'\b(?:without|exclude|remove|no|avoid)\s+(?:any\s+)?'+re.escape(term)+r'(?!\w)', text):
                if canonical not in excluded:
                    excluded.append(canonical)
            elif canonical not in positive:
                positive.append(canonical)
        # Prefer longer matched ingredient names over their embedded substrings.
        positive = [x for x in positive if not any(x != y and x in y for y in positive)]
        return {'intent':'search','ingredients':positive[:10], 'excluded':excluded[:10],
                'cuisine':cuisine,'diet':diet,'max_minutes':minutes}

    def extract(self, utterance):
        fallback = self._rules(utterance)
        if self.pipe is None or fallback.get('intent') in ('reset','help'):
            return fallback
        instruction = ('Extract recipe preferences from the user message. Return ONLY JSON with keys '
                       'intent, ingredients, excluded, cuisine, diet, max_minutes. '
                       'Use null for absent scalar values and [] for absent arrays. '
                       'Intent is search, reset or help. Do not invent preferences.')
        try:
            messages = [{'role':'system','content':instruction},
                        {'role':'user','content':utterance}]
            result = self.pipe(messages, max_new_tokens=180, do_sample=False)
            generated = result[0]['generated_text']
            answer = generated[-1]['content'] if isinstance(generated, list) else generated
            match = re.search(r'\{.*?\}', answer, flags=re.S)
            obj = json.loads(match.group()) if match else {}
            clean = {'intent':obj.get('intent','search')}
            for key in ('ingredients','excluded'):
                values = obj.get(key) or []
                if isinstance(values, str): values = [values]
                clean[key] = list(dict.fromkeys(normalize(x) for x in values
                                if isinstance(x, str) and normalize(x) in self.vocabulary))[:10]
            for key, allowed in (('cuisine',CUISINES),('diet',DIETS)):
                value = normalize(obj[key]) if isinstance(obj.get(key),str) else None
                clean[key] = value if value in allowed else None
            try:
                val = obj.get('max_minutes')
                clean['max_minutes'] = int(val) if val is not None and 0 < int(val) < 10000 else None
            except (ValueError,TypeError):
                clean['max_minutes'] = None
            # If model fails to recognize a clear constraint, use the rule-based result.
            for key in ('ingredients','excluded','cuisine','diet','max_minutes'):
                if not clean.get(key): clean[key] = fallback.get(key)
            return clean
        except (ValueError,TypeError,KeyError,IndexError,AttributeError):
            return fallback


class SavoraBot:
    def __init__(self, data_dir='.', use_qwen=False,
                 model_id='Qwen/Qwen2.5-1.5B-Instruct', max_rows=None):
        self.store = RecipeStore(Path(data_dir)/'RAW_recipes.csv', max_rows=max_rows)
        self.nlu = NLU(self.store.vocabulary, use_qwen=use_qwen, model_id=model_id)
        self.state = Slots()
        self.last_results = []

    def reset(self):
        self.state = Slots()
        self.last_results = []

    def respond(self, message):
        if not str(message).strip(): return 'Please describe what you would like to cook.'
        obj = self.nlu.extract(message)
        if obj.get('intent') == 'reset':
            self.reset()
            return 'Preferences cleared. What would you like to cook?'
        if obj.get('intent') == 'help':
            return ('Try: "Vegetarian Indian recipe with potato under 30 minutes". '
                    'Follow up with "under 20 minutes", "without onion", or "reset".')
        for key in ('cuisine','diet','max_minutes'):
            if obj.get(key) is not None: setattr(self.state,key,obj[key])
        for item in obj.get('excluded',[]):
            if item not in self.state.excluded: self.state.excluded.append(item)
            if item in self.state.ingredients: self.state.ingredients.remove(item)
        for item in obj.get('ingredients',[]):
            if item not in self.state.ingredients: self.state.ingredients.append(item)
            if item in self.state.excluded: self.state.excluded.remove(item)
        if not any((self.state.ingredients,self.state.excluded,self.state.cuisine,
                    self.state.diet,self.state.max_minutes)):
            return 'What ingredients, cuisine, diet, or cooking time would you prefer?'
        self.last_results = self.store.search(self.state)
        if not self.last_results:
            return ('No exact matches for these preferences. Try a longer cooking time, '
                    'fewer required ingredients, or say "reset".')
        lines = ['Here are recipes matching your current preferences:']
        for index, rec in enumerate(self.last_results,1):
            mins = f"{rec['minutes']:g}" if pd.notna(rec['minutes']) else '?'
            lines.append(f"{index}. **{str(rec['name']).title()}** — {mins} min (recipe ID {rec['id']})")
            lines.append('   Ingredients: '+', '.join(map(str,rec['ingredients'][:8])))
        lines.append('Refine your search with a follow-up, or type "reset".')
        return '\n'.join(lines)

    def recipe_details(self, result_number=1):
        if not 1 <= result_number <= len(self.last_results):
            return 'Choose a recipe number from your most recent results.'
        rec = self.last_results[result_number-1]
        steps = '\n'.join(f'{i}. {step}' for i,step in enumerate(rec['steps'],1))
        return (f"{rec['name'].title()}\nIngredients: {', '.join(rec['ingredients'])}\n"
                f"Instructions:\n{steps}")


def launch_gradio(bot, share=False):
    import gradio as gr
    # Separate state per browser conversation rather than sharing one global bot.
    def chat(message, history, state):
        if state is None: state = Slots()
        session_bot = copy.copy(bot)
        session_bot.state = state
        response = session_bot.respond(message)
        return response, session_bot.state
    with gr.Blocks(title='Savora Recipe Assistant') as demo:
        gr.Markdown('# Savora — Conversational Recipe Assistant')
        gr.Markdown('Ask for recipes by ingredients, cuisine, diet, or cooking time.')
        state = gr.State(None)
        chatbox = gr.Chatbot()
        user_input = gr.Textbox(label='Your message', placeholder='Vegetarian Indian potato dish under 30 minutes')
        def submit(message, history, session_state):
            reply, session_state = chat(message, history, session_state)
            history = history + [{'role':'user','content':message},{'role':'assistant','content':reply}]
            return '', history, session_state
        user_input.submit(submit,[user_input,chatbox,state],[user_input,chatbox,state])
        gr.Button('Reset chat').click(lambda: ('',[],None),None,[user_input,chatbox,state])
    return demo.launch(share=share, inbrowser=False)


## Conversational configuration and portable dataset discovery

Searches the repository and common dataset locations without assuming any person's Windows username. If `RAW_recipes.csv` is unavailable, NLU and normalization checks still run. A large Food.com dataset is **not** bundled with this notebook.

In [56]:
import os
from pathlib import Path
USE_QWEN = False  # Explicitly opt in: downloads a model and requires substantial memory.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_ROWS = 10000  # Set None for full dataset when resources permit.

def locate_recipe_dataset():
    explicit = os.environ.get("SAVORA_DATA_DIR", "")
    roots = [Path(explicit)] if explicit else []
    repo = find_repo_root()
    if repo is not None:
        roots += [repo / "Data" / "Food.com", repo / "data" / "Food.com", repo]
    roots += [Path.cwd(), Path.cwd() / "Data" / "Food.com", Path.home() / "Downloads" / "Food.com", Path.home() / "Downloads", Path.home() / "Documents"]
    for root in roots:
        candidate = root / "RAW_recipes.csv"
        if candidate.is_file():
            return candidate
    return None

RECIPE_CSV = locate_recipe_dataset()
print("Recipe dataset:", RECIPE_CSV or "not found — conversational dataset tests will be skipped")


Recipe dataset: not found — conversational dataset tests will be skipped


In [57]:
# Run without the Food.com dataset: entity normalization and rule-based NLU.
assert normalize("brinjal") == "eggplant"
assert normalize("capsicum") == "bell pepper"
standalone_nlu = NLU({"potato", "eggplant", "bell pepper"}, use_qwen=False)
print("NLU smoke test:", standalone_nlu.extract("I want a vegetarian Indian potato dish under 30 minutes"))
print("Entity normalization smoke tests passed")


NLU smoke test: {'intent': 'search', 'ingredients': ['potato'], 'excluded': [], 'cuisine': 'indian', 'diet': 'vegetarian', 'max_minutes': 30}
Entity normalization smoke tests passed


In [58]:
# Full chatbot requires the original Food.com RAW_recipes.csv.
chatbot = None
if RECIPE_CSV is not None:
    chatbot = SavoraBot(data_dir=RECIPE_CSV.parent, use_qwen=USE_QWEN, model_id=MODEL_ID, max_rows=MAX_ROWS)
    print("Recipes loaded:", len(chatbot.store.df))
    print("Chatbot example:", chatbot.respond("vegetarian Indian dish with potato under 30 minutes"))
else:
    print("Skipped dataset-backed chatbot. Set SAVORA_DATA_DIR to the folder containing RAW_recipes.csv and rerun.")


Skipped dataset-backed chatbot. Set SAVORA_DATA_DIR to the folder containing RAW_recipes.csv and rerun.


In [59]:
%pip install --upgrade gradio

Note: you may need to restart the kernel to use updated packages.


In [60]:
import gradio as gr
import inspect

print("Gradio version:", gr.__version__)
print("Chatbot parameters:", inspect.signature(gr.Chatbot))

Gradio version: 6.30.0
Chatbot parameters: (value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_classes: 'list[str] | str | None' = None, autoscroll: 'bool' = True, render: 'bool' = True, key: 'int | str | tuple[int | str, ...] | None' = None, preserved_by_key: 'list[str] | str | None' = 'value', height: 'int | str | None' = 400, resizable: 'bool' = False, max_height: 'int | str | None' = None, min_height: 'int | str | None' = None, editable: "Literal['user', 'all'] | None" = None, latex_delimiters: 'list[dict[str, str | bool]] | None' = None, rtl: 'bool' = False, buttons: "list[Literal['share', 'copy', 'copy_all'] | Button] | None" = N

In [61]:
import gradio as gr

def savora_chat(message, history):
    return str(savora_bot.respond(message))

demo = gr.ChatInterface(
    fn=savora_chat,
    title="Savora AI Meal Planner",
    description="Ask Savora for recipe recommendations."
)

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Optional interactive chat

These cells are opt-in and require `ipywidgets` or `gradio` installed in the active notebook kernel. They are intentionally not auto-launched.

In [62]:
if chatbot is not None:    
   launch_gradio(savora_bot)

if RECIPE_CSV is not None:
    qwen_chatbot = SavoraBot(data_dir=RECIPE_CSV.parent, use_qwen=True, model_id=MODEL_ID, max_rows=MAX_ROWS)
    print(qwen_chatbot.nlu.extract("Find vegetarian Indian recipes without onion"))


In [63]:
print("Chatbot type:", type(chatbot))
print("Has respond method:", hasattr(chatbot, "respond"))
print("Has SavoraBot:", "SavoraBot" in globals())

Chatbot type: <class 'NoneType'>
Has respond method: False
Has SavoraBot: True


In [64]:
for name in ["SavoraBot", "RECIPE_CSV", "MODEL_ID", "MAX_ROWS", "savora_bot"]:
    print(name, ":", name in globals())

SavoraBot : True
RECIPE_CSV : True
MODEL_ID : True
MAX_ROWS : True
savora_bot : True


In [65]:
from pathlib import Path

RECIPE_CSV = Path(
    r"C:\Users\Aryaman\OneDrive\Desktop\MR2 Workshop\Savora-AI-Meal-Planner-GC-1-1\Food.com\RAW_recipes.csv"
)

if not RECIPE_CSV.is_file():
    raise FileNotFoundError(f"Dataset not found: {RECIPE_CSV}")

savora_bot = SavoraBot(
    data_dir=RECIPE_CSV.parent,
    use_qwen=False,
    model_id=MODEL_ID,
    max_rows=MAX_ROWS
)

print("SavoraBot initialized successfully!")
print("Respond method:", hasattr(savora_bot, "respond"))

SavoraBot initialized successfully!
Respond method: True


In [66]:
print("SavoraBot ready:", isinstance(savora_bot, SavoraBot))
print("Respond method:", hasattr(savora_bot, "respond"))

SavoraBot ready: True
Respond method: True


In [67]:
savora_bot = SavoraBot(
    data_dir=RECIPE_CSV.parent,
    use_qwen=False,
    model_id=MODEL_ID,
    max_rows=MAX_ROWS
)

print("SavoraBot initialized successfully!")
print("Bot type:", type(savora_bot))

SavoraBot initialized successfully!
Bot type: <class '__main__.SavoraBot'>


In [68]:
response = savora_bot.respond(
    "Suggest a vegetarian Indian recipe using potatoes under 30 minutes"
)

print(response)

Here are recipes matching your current preferences:
1. **Tide Me Over   Indian Chaat  Simple Veggie Salad** — 2 min (recipe ID 63793)
   Ingredients: cucumber, potato, chopped tomato, plain yogurt, chat masala, paprika, salt
2. **Aloo  Potato  Pakora** — 10 min (recipe ID 110215)
   Ingredients: potatoes, corn oil, besan, water, red chili powder, cumin powder, garlic, ginger
3. **Aloo Curry  Potato Curry** — 25 min (recipe ID 11512)
   Ingredients: potatoes, onions, garlic, ginger, tomatoes, green chilies, salt, turmeric
4. **Aaloo Chaat  Tangy Potatoes** — 25 min (recipe ID 96178)
   Ingredients: potato, red chili powder, chat masala, coriander powder, onion, salt, tamarind pulp, oil
5. **Aloo Baigain  Potato And Eggplant  Aubergine** — 25 min (recipe ID 149969)
   Ingredients: potato, eggplants, red chili powder, cumin, coriander powder, turmeric powder, salt, garlic paste
Refine your search with a follow-up, or type "reset".


In [69]:
launch_gradio(savora_bot)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [70]:
print("SavoraBot respond method:", hasattr(SavoraBot, "respond"))
print("Dataset:", RECIPE_CSV)

SavoraBot respond method: True
Dataset: C:\Users\Aryaman\OneDrive\Desktop\MR2 Workshop\Savora-AI-Meal-Planner-GC-1-1\Food.com\RAW_recipes.csv


In [71]:
print("Recipe columns:")
print(savora_bot.store.df.columns.tolist())

print("\nMinutes column exists:")
print("minutes" in savora_bot.store.df.columns)

Recipe columns:
['name', 'id', 'minutes', 'tags', 'steps', 'description', 'ingredients', 'ingredient_set', 'tag_set']

Minutes column exists:
True


In [72]:
import traceback

try:
    response = savora_bot.respond(
        "Suggest a vegetarian Indian recipe using potatoes under 30 minutes"
    )
    print(response)
except Exception:
    traceback.print_exc()

Here are recipes matching your current preferences:
1. **Tide Me Over   Indian Chaat  Simple Veggie Salad** — 2 min (recipe ID 63793)
   Ingredients: cucumber, potato, chopped tomato, plain yogurt, chat masala, paprika, salt
2. **Aloo  Potato  Pakora** — 10 min (recipe ID 110215)
   Ingredients: potatoes, corn oil, besan, water, red chili powder, cumin powder, garlic, ginger
3. **Aloo Curry  Potato Curry** — 25 min (recipe ID 11512)
   Ingredients: potatoes, onions, garlic, ginger, tomatoes, green chilies, salt, turmeric
4. **Aaloo Chaat  Tangy Potatoes** — 25 min (recipe ID 96178)
   Ingredients: potato, red chili powder, chat masala, coriander powder, onion, salt, tamarind pulp, oil
5. **Aloo Baigain  Potato And Eggplant  Aubergine** — 25 min (recipe ID 149969)
   Ingredients: potato, eggplants, red chili powder, cumin, coriander powder, turmeric powder, salt, garlic paste
Refine your search with a follow-up, or type "reset".


In [73]:
import ipywidgets as widgets
from IPython.display import display

button = widgets.Button(description="Test Savora")
display(button)

Button(description='Test Savora', style=ButtonStyle())

## Integration boundary and limitations

- The original `SavoraBot` and its `Slots` are conversational state, **not** the validated `UserProfile` or authoritative `RuleContext` from the reasoning package.
- Convert normalized entities and slot values to a versioned, validated profile through a dedicated backend adapter; never silently reinterpret unknown allergens as safe.
- Only call `filter_candidates`, `evaluate_combination`, and `solve_daily_plan` with real validated snapshot DTOs and accepted KB/source provenance.
- The repository snapshot does **not** provide a validated 7-day CSP optimizer or fully implemented meal-plan backend routes. Contract placeholders must not be presented as actual recommendations.
- `MAX_ROWS` is a development performance setting; using a partial dataset changes retrieval coverage. Qwen is opt-in, not necessary for the rule-based smoke tests.
- To avoid disrupting your teammate's work, keep this notebook in a separate folder and integrate through a pull request after interface agreement.